# `ch3-measurability` — end-to-end analysis

Step-by-step replay of the Chapter 3 experimental pipeline. Each cell reads a JSON or `.npy` file produced by the corresponding script in `scripts/` and renders a plot or table. Run top-to-bottom to regenerate every diagnostic figure and every table for the thesis.

**Inputs.** All sealed in `inputs/` with SHA-256 in `manifest.json`. The 364 core terms (curated post-BLP, K≥4 in HK Cap. enacted post-1989), 9.045 background terms (legalish residual), and 100 control terms (everyday vocabulary).

**Algorithms.** Re-importable from `scripts/_lib.py` (self-contained: RDM, Mantel, block bootstrap, Mann-Whitney, Holm correction, EmbeddingClient).

**Outputs.** `experiment_1_structure/results_{bare,attested}/`, `experiment_2_axes/results_{bare,attested}/`, `ext/A..Z/`, plus consolidated markdown in `reports/`.

**Statistical parameters.** B(Mantel) = 10000, B(block bootstrap) = 10000, seed = 42, CPU float32.


## 1. Setup

In [ ]:
import json, sys
from pathlib import Path
from itertools import combinations, product
import numpy as np, pandas as pd, yaml
import matplotlib.pyplot as plt

RUN = Path.cwd().resolve().parent
REPO_ROOT = RUN.parents[1]
sys.path.insert(0, str(RUN / 'scripts'))
import _lib

with (RUN / 'config.yaml').open() as fh:
    cfg = yaml.safe_load(fh)

EMB = RUN / 'embeddings'
EXP1 = RUN / 'experiment_1_structure'
EXP2 = RUN / 'experiment_2_axes'
EXT = RUN / 'ext'
PLOTS = RUN / 'reports/diagnostic_plots'
PLOTS.mkdir(exist_ok=True)

exp1_bare = json.loads((EXP1 / 'results_bare/experiment_1_results.json').read_text())
exp1_att  = json.loads((EXP1 / 'results_attested/experiment_1_results.json').read_text())
exp2_bare = json.loads((EXP2 / 'results_bare/experiment_2_results.json').read_text())
exp2_att  = json.loads((EXP2 / 'results_attested/experiment_2_results.json').read_text())
print(f"N={exp1_bare['meta']['n_terms']}, B_mantel={exp1_bare['meta']['n_perm']}, B_boot={exp1_bare['meta']['n_boot']}, seed={exp1_bare['meta']['seed']}")

## 2. Pool description

The three term pools (core, background, control), distribution by legal domain for the core, attested-context coverage.

In [ ]:
from collections import Counter
core_index = json.loads((EMB / 'index.json').read_text())
bg_index = json.loads((EMB / 'bg/index.json').read_text())
ctrl_index = json.loads((EMB / 'control_bare/index.json').read_text())

print(f'core terms       : {len(core_index)}')
print(f'background terms : {len(bg_index)}')
print(f'control terms    : {len(ctrl_index)}')
print()
print('core distribution by legal domain:')
for d, n in sorted(Counter(t['domain'] for t in core_index).items()):
    print(f'  {d:18s}  {n}')
print()
k_min_core = json.loads((RUN / 'inputs/context_coverage_snapshot.json').read_text())['per_term']
kmins = [v['k_min'] for v in k_min_core.values()]
print(f'core K_min: min={min(kmins)}, max={max(kmins)}, mean={np.mean(kmins):.2f}, K_min≥4: {sum(1 for k in kmins if k >= 4)}/{len(kmins)}')

## 3. Encoding diagnostic — 10 models × {bare, attested}

Confirms L2-norm = 1 ± 1e-6 on every embedding matrix, lists dim and attested coverage.

In [ ]:
models = sorted(p.name for p in EMB.iterdir() if p.is_dir() and p.name not in ('bg', 'control_bare'))
rows = []
for m in models:
    meta = json.loads((EMB / m / 'meta.json').read_text())
    cov  = json.loads((EMB / m / 'coverage.json').read_text())
    bare = np.load(EMB / m / 'vecs_bare.npy')
    att  = np.load(EMB / m / 'vecs_attested.npy')
    rows.append({
        'model': m, 'lang': meta['lang'], 'dim': meta['dim'],
        'bare_norm_range': f"[{np.linalg.norm(bare, axis=1).min():.5f}, {np.linalg.norm(bare, axis=1).max():.5f}]",
        'attested_norm_range': f"[{np.linalg.norm(att, axis=1).min():.5f}, {np.linalg.norm(att, axis=1).max():.5f}]",
        'attested_min_K': cov['min_n_attested'],
        'attested_mean_K': round(cov['mean_n_attested'], 2),
        'attested_K_below_4': cov['n_terms_k_lt_4'],
    })
pd.DataFrame(rows)

## 4. Experiment 1 (Distance Structure, §3.1)

### 4.1 §3.1.1 — Intra-domain vs inter-domain (Mann-Whitney, 3 WEIRD models)

In [ ]:
rows = []
for variant_name, exp1 in [('bare', exp1_bare), ('attested', exp1_att)]:
    for label, r in exp1['section_311']['per_model'].items():
        rows.append({
            'variant': variant_name, 'model': label,
            'intra median': r['median_x'], 'inter median': r['median_y'],
            'effect r': r['effect_r'], 'p_value': r['p_value'],
        })
pd.DataFrame(rows)

### 4.2 §3.1.1 — Legal-vs-control (bare only, N_control=100)

Mann-Whitney U one-sided: do legal-legal distances form a tighter cluster than legal-control distances? Effect size r = rank-biserial.

In [ ]:
lvc = exp1_bare['section_311_legal_vs_control']['per_model']
df_lvc = pd.DataFrame([
    {'model': m, 'legal med': v['median_x'], 'ctrl med': v['median_y'],
     'effect r': v['effect_r'], 'p_value': v['p_value']}
    for m, v in lvc.items()
])
n_pos = (df_lvc['effect_r'] > 0).sum() & (df_lvc['p_value'] < 0.05).sum()
print(f"{n_pos}/{len(df_lvc)} models confirm signal (r>0 & p<0.05)")
df_lvc.sort_values('effect r', ascending=False)

### 4.3 §3.1.2 — Domain topology consensus (7×7 matrix)

Mean cosine-distance matrix between domains. Diagonal = intra-domain mean distance; off-diagonal = inter-domain mean. Consensus across the 10 models.

In [ ]:
def consensus_topology(exp1):
    mats, doms = [], None
    for r in exp1['section_312']['per_model'].values():
        mats.append(np.array(r['matrix']))
        doms = r['domains']
    return np.array(mats).mean(axis=0), doms

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
for i, (variant, exp1) in enumerate([('bare', exp1_bare), ('attested', exp1_att)]):
    cons, doms = consensus_topology(exp1)
    im = ax[i].imshow(cons, cmap='magma')
    ax[i].set_xticks(range(len(doms))); ax[i].set_yticks(range(len(doms)))
    ax[i].set_xticklabels(doms, rotation=45, ha='right')
    ax[i].set_yticklabels(doms)
    ax[i].set_title(f'{variant} — consensus (n=10 models)')
    plt.colorbar(im, ax=ax[i], shrink=0.7, label='mean cosine distance')
fig.tight_layout()
fig.savefig(PLOTS / 'domain_topology_consensus.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.4 §3.1.3 — RSA forest (17 pre-registered model pairs)

Spearman ρ on RDM upper triangles, with Mantel test (B=10000) and block bootstrap CI (B=10000). Holm-Bonferroni correction across all 17 pairs.

In [ ]:
def collect_rsa(exp1, variant):
    rows = []
    s = exp1['section_313']
    for group in ['within_weird', 'within_sinic', 'cross_tradition', 'within_bilingual']:
        for e in s.get(group, []):
            rows.append({'variant': variant, 'group': group,
                         'pair': f"{e['model_a']} × {e['model_b']}",
                         'rho': e['rho'], 'ci_low': e['ci_low'], 'ci_high': e['ci_high'],
                         'p_holm': e['p_holm']})
    return pd.DataFrame(rows)

rsa_df = pd.concat([collect_rsa(exp1_bare, 'bare'), collect_rsa(exp1_att, 'attested')], ignore_index=True)
group_order = ['within_weird', 'within_sinic', 'cross_tradition', 'within_bilingual']
group_color = {'within_weird': '#0072B2', 'within_sinic': '#D55E00',
               'cross_tradition': '#009E73', 'within_bilingual': '#CC79A7'}
fig, axes = plt.subplots(1, 2, figsize=(13, 7), sharex=True)
for ax_i, variant in zip(axes, ['bare', 'attested']):
    sub = rsa_df[rsa_df['variant'] == variant].copy()
    sub['gidx'] = sub['group'].map({g: i for i, g in enumerate(group_order)})
    sub = sub.sort_values(['gidx', 'rho']).reset_index(drop=True)
    ys = np.arange(len(sub))
    ax_i.errorbar(sub['rho'], ys, xerr=[sub['rho']-sub['ci_low'], sub['ci_high']-sub['rho']],
                  fmt='o', ecolor='#888', mfc='white', ms=7, capsize=3)
    for y, rho, g in zip(ys, sub['rho'], sub['group']):
        ax_i.plot(rho, y, 'o', color=group_color[g], ms=8)
    ax_i.set_yticks(ys); ax_i.set_yticklabels(sub['pair'], fontsize=8)
    ax_i.axvline(0, color='#aaa', ls='--', lw=0.6)
    ax_i.set_title(variant); ax_i.set_xlabel('Spearman ρ')
    ax_i.grid(axis='x', alpha=0.3)
fig.tight_layout()
fig.savefig(PLOTS / 'rsa_forest.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
for variant_name, exp1 in [('bare', exp1_bare), ('attested', exp1_att)]:
    s = exp1['section_313']['summary']
    print(f"--- {variant_name} ---")
    print(f"  within-WEIRD ρ̄  = {s['mean_rho_within_weird']:.3f}")
    print(f"  within-Sinic ρ̄  = {s['mean_rho_within_sinic']:.3f}")
    print(f"  cross ρ̄          = {s['mean_rho_cross_tradition']:.3f}")
    print(f"  Δρ_sym           = {s['delta_rho_symmetric']:.3f}")
    if 'mean_rho_within_bilingual' in s:
        print(f"  within-bilingual ρ̄ = {s['mean_rho_within_bilingual']:.3f}")

## 5. Experiment 2 (Value-axis projection, §3.2)

### 5.1 §3.2.4 — Per-axis cross-tradition ρ̄

In [ ]:
summary = (pd.DataFrame({
    'bare': exp2_bare['section_324']['cross_rho_mean_per_axis'],
    'attested': exp2_att['section_324']['cross_rho_mean_per_axis'],
}).sort_values('attested'))
summary

### 5.2 §3.2.4 — Ranking (most divergent → least)

In [ ]:
for i, e in enumerate(exp2_att['section_324']['ranking_most_divergent_first'], 1):
    print(f"{i}. {e['axis']:25s}  ρ̄ = {e['mean_cross_rho']:.3f}")

### 5.3 §3.2.5 — Top-5 most-divergent terms per axis

In [ ]:
for ax_name in exp2_att['meta']['axes']:
    print(f"\n→ {ax_name}")
    for t in exp2_att['section_325'][ax_name]['top_K_divergent'][:5]:
        print(f"   {t['en'][:35]:35s}  W̄={t['w_score']:+.3f}  S̄={t['s_score']:+.3f}  Δ={t['delta']:+.3f}")

## 6. Extension A — k-NN background-domain assignment

9.045 background terms routed to one of 7 legal domains via k=7 nearest neighbours in the 364 core.

In [ ]:
a = json.loads((EXT / 'A_bg_knn/background_assignments.json').read_text())
print('Domain distribution:')
for d, n in sorted(a['meta']['domain_distribution'].items(), key=lambda x: -x[1]):
    print(f'  {d:18s}  {n}')
print(f"\nconfidence mean={a['meta']['confidence_mean']:.3f}, median={a['meta']['confidence_median']:.3f}, p90={a['meta']['confidence_high_decile']:.3f}")

## 7. Extension D — Δρ_sym vs %bg robustness curve

Inject increasing percentages of bg into the 364 core, recompute Δρ_sym attested. Does the cross-tradition gap survive pool perturbation?

In [ ]:
d = json.loads((EXT / 'D_robustness/robustness_curve.json').read_text())
df_d = pd.DataFrame(d['results'])
fig, ax = plt.subplots(figsize=(7, 5))
ax.errorbar(df_d['p_bg']*100, df_d['mean_delta_sym'],
            yerr=[df_d['mean_delta_sym']-df_d['ci_low_delta_sym'],
                  df_d['ci_high_delta_sym']-df_d['mean_delta_sym']],
            fmt='o-', color='#009E73', capsize=5, lw=2, ms=8)
ax.axhline(0.4, color='red', ls=':', lw=0.8, label='verification gate threshold (Δρ_sym ≥ 0.4)')
ax.set_xlabel('% background terms injected')
ax.set_ylabel('Δρ_sym attested')
ax.set_title('Robustness of Δρ_sym under bg injection (attested)')
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(PLOTS / 'robustness_vs_bg.png', dpi=150, bbox_inches='tight')
plt.show()
df_d[['p_bg', 'n_core', 'n_bg', 'mean_delta_sym', 'std_delta_sym', 'ci_low_delta_sym', 'ci_high_delta_sym']]

## 8. Extension E — Out-of-sample axes projection

The 6 axes built on the core 364 are projected onto the 9.045 bg. Coherence per k-NN-assigned domain reported.

In [ ]:
e = json.loads((EXT / 'E_axes_oos/coherence.json').read_text())
# Show mean bg-score per domain for one representative model + axis
label = e['meta']['labels'][0]
ax_name = e['meta']['axes'][0]
print(f'Example: {label} on axis `{ax_name}` — mean score of bg per assigned domain:')
for dom, stats in sorted(e['per_model_per_axis_per_domain'][label][ax_name].items(), key=lambda x: x[1]['mean']):
    print(f'  {dom:18s}  n={stats["n"]:5d}  mean={stats["mean"]:+.3f}  std={stats["std"]:.3f}')

## 9. Extension F — Confidence-stratified Δρ_sym

Inject 91 bg into the 364 core, drawn from the top vs bottom decile of k-NN confidence, vs random control. 20 replicates each.

In [ ]:
f = json.loads((EXT / 'F_confidence/confidence_strata.json').read_text())
base = f['baseline_core_only']['delta_sym']
for label, key in [('baseline (core only)', 'baseline_core_only'),
                   ('high-conf bg', 'high_confidence_bg_injected'),
                   ('low-conf bg', 'low_confidence_bg_injected'),
                   ('random control bg', 'random_control_bg_injected')]:
    if key == 'baseline_core_only':
        print(f"  {label:25s}  Δρ_sym = {base:.3f}")
    else:
        v = f[key]
        delta = v['mean_delta_sym'] - base
        print(f"  {label:25s}  Δρ_sym = {v['mean_delta_sym']:.3f} (Δ vs baseline {delta:+.3f}, std={v['std_delta_sym']:.3f})")

## 10. Extension G — Automated false-friends with bilingual control

For each bg with K_en≥2 AND K_zh≥2 (4.156 candidates), cross-encoder cosine (BGE-EN × BGE-ZH) vs bilingual-control cosine (BGE-M3-EN × BGE-M3-ZH, same encoder both sides).

In [ ]:
g = json.loads((EXT / 'G_false_friends/false_friends.json').read_text())
en_m, zh_m = g['meta']['en_model'], g['meta']['zh_model']
bi = g['meta']['bilingual_pair']
cross_key = f'cos_{en_m}_vs_{zh_m}'
bi_key = f'cos_{bi[0]}_vs_{bi[1]}'
df_g = pd.DataFrame(g['rows'])[['en', 'zh', 'k_en', 'k_zh', cross_key, bi_key]]
df_g.columns = ['en', 'zh', 'K_en', 'K_zh', 'cross (BGE)', 'bilingual (BGE-M3)']
df_g.head(20)

## 11. Extension H — K saturation curve

ρ_cross attested as a function of bg K_min bucket. Below K=4 the signal is unstable; at K=1 it is anti-correlated.

In [ ]:
h = json.loads((EXT / 'H_K_saturation/k_saturation.json').read_text())
rows = [{'K bucket': b['K_bucket'], 'n_bg': b['n_bg'], 'ρ_cross': b['mean_rho_cross']} for b in h['buckets']]
rows.append({'K bucket': 'core (4-8)', 'n_bg': 364, 'ρ_cross': h['meta']['core_reference_attested_cross_rho']})
df_h = pd.DataFrame(rows)
df_h

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
buckets_x = ['1', '2', '3', '4-7', '8', 'core 4-8']
rhos = df_h['ρ_cross'].tolist()
ax.bar(range(len(rhos)), rhos, color=['#D55E00' if r < 0.1 else '#56B4E9' for r in rhos], alpha=0.85)
ax.axhline(0, color='black', lw=0.8)
ax.axhline(0.246, color='#009E73', ls=':', lw=0.8, label='headline ρ_cross core attested')
ax.set_xticks(range(len(buckets_x))); ax.set_xticklabels(buckets_x)
ax.set_xlabel('K_min bucket'); ax.set_ylabel('ρ_cross attested (BGE-EN × BGE-ZH)')
ax.set_title('K saturation curve — empirical justification for K≥4 threshold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig(PLOTS / 'k_saturation.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Extension X — Δρ_sym vs %control (dual of D, bare)

Mirror of D using control terms (non-legal) instead of bg. Expectation: Δρ_sym bare declines with %control. Bare-only by design (no attested for control).

In [ ]:
x = json.loads((EXT / 'X_control_robustness/control_robustness_curve.json').read_text())
df_x = pd.DataFrame(x['results'])
fig, ax = plt.subplots(figsize=(7, 5))
ax.errorbar(df_x['p_control']*100, df_x['mean_delta_sym'],
            yerr=df_x['std_delta_sym'], fmt='o-', color='#D55E00', capsize=5, lw=2, ms=8)
ax.set_xlabel('% control terms injected')
ax.set_ylabel('Δρ_sym bare')
ax.set_title('Discriminative direction — Δρ_sym falls with non-legal injection')
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(PLOTS / 'robustness_vs_control.png', dpi=150, bbox_inches='tight')
plt.show()
df_x[['p_control', 'n_core', 'n_control', 'mean_delta_sym', 'std_delta_sym']]

## 13. Extension Y — Cross-tradition ρ on control-only (CRUCIAL CAVEAT)

**The most important methodological result.** Δρ_sym bare on the 100 control terms (everyday vocabulary) is statistically indistinguishable from Δρ_sym bare on the 364 core. The bare signal is *encoder-tradition shaped*, NOT *legal-tradition shaped*. The legal-attestation contribution is the **attested-bare gap on the core**: 0.378 = 0.543 − 0.165.

In [ ]:
y = json.loads((EXT / 'Y_control_only/control_only_rsa.json').read_text())
s, cmp_ = y['summary'], y['comparison']
print('Control-only pool (N=100, bare):')
print(f"  within-WEIRD ρ̄    = {s['mean_rho_within_weird']:.3f}")
print(f"  within-Sinic ρ̄    = {s['mean_rho_within_sinic']:.3f}")
print(f"  cross-tradition ρ̄ = {s['mean_rho_cross_tradition']:.3f}")
print(f"  Δρ_sym             = {s['delta_rho_symmetric']:.3f}")
print()
print('Comparison:')
print(f"  Δρ_sym 364 core bare     = {cmp_['delta_sym_core_bare_run4']:.3f}")
print(f"  Δρ_sym 364 core attested = {cmp_['delta_sym_core_attested_run4']:.3f}")
print(f"  Δρ_sym 100 control bare  = {cmp_['delta_sym_control_bare']:.3f}  ← indistinguishable from core bare")
print()
gap = cmp_['delta_sym_core_attested_run4'] - cmp_['delta_sym_core_bare_run4']
print(f'>> LEGAL signal = attested-bare gap on core = {gap:.3f}')
print(f'>> Encoder-tradition baseline (shared with non-legal vocabulary) ≈ 0.16')

## 14. Extension Z — Three-tier distance hierarchy

In [ ]:
z = json.loads((EXT / 'Z_tier_hierarchy/tier_hierarchy.json').read_text())
df_z = pd.DataFrame([
    {'model': label, 'core×core': v['median']['core_core'],
     'core×bg': v['median']['core_bg'], 'core×control': v['median']['core_control'],
     'monotonic': '✓' if v['monotonic_hierarchy'] else '✗'}
    for label, v in z['per_model'].items()
])
df_z

In [ ]:
n_mono = z['meta']['n_models_with_monotonic_hierarchy']
n_total = z['meta']['n_models']
print(f'{n_mono}/{n_total} models satisfy median(core×core) < median(core×bg) < median(core×control).')
print(f'In {n_total-n_mono}/{n_total}, bg are farther from core than control. Tier classification is curative, not geometric.')

## 15. Comparison vs run #3 (Firthian 327)

Δ = run #4 − run #3 on the attested column. Used to argue the stability of Δρ_sym across two independently-curated pools.

In [ ]:
RUN3 = {
    'within-WEIRD ρ̄ attested': 0.760,
    'within-Sinic ρ̄ attested': 0.845,
    'cross ρ̄ attested': 0.259,
    'Δρ_sym attested': 0.541,
    'within-bilingual ρ̄ attested': 0.340,
}
s4 = exp1_att['section_313']['summary']
RUN4 = {
    'within-WEIRD ρ̄ attested': s4['mean_rho_within_weird'],
    'within-Sinic ρ̄ attested': s4['mean_rho_within_sinic'],
    'cross ρ̄ attested': s4['mean_rho_cross_tradition'],
    'Δρ_sym attested': s4['delta_rho_symmetric'],
    'within-bilingual ρ̄ attested': s4.get('mean_rho_within_bilingual'),
}
pd.DataFrame({
    'run #3 (327 Firthian)': RUN3,
    'run #4 (364 post-BLP)': RUN4,
    'Δ': {k: round(RUN4[k] - RUN3[k], 3) for k in RUN3},
})

## 16. Conclusions for the thesis

**Three anchor results.**

1. **Δρ_sym is structurally stable** under pool re-curation. Run #3 (327 Firthian) → Δρ_sym attested = 0.541; run #4 (364 post-BLP) → 0.543; extension D (sampled mix levels 0%–75% bg) → 0.535-0.590. Seven distinct snapshots, same gap.

2. **The gap lives in the cross-tradition encoder pair, not in the term and not in any single encoder.** Extension G shows ≈50 same-lemma bg terms with negative WEIRD×Sinic cross-encoder cosine and +0.5 to +0.75 bilingual cosine — a single bilingual encoder (BGE-M3) reads the two sides as the same vector while two tradition-tuned encoders read them as opposing vectors.

3. **The legal-meaning signal is the attested-bare gap, not the attested absolute.** Extension Y shows Δρ_sym bare = 0.156 on the 100 control terms (everyday vocabulary), indistinguishable from 0.165 on the 364 core. Δρ_sym attested = 0.543 by itself is methodologically ambiguous; the *legal* contribution is the gap **0.543 − 0.165 = 0.378** against the encoder-tradition baseline.

**Qualifying caveats** (§4.2):

- Extension Z: tier classification (core/bg/control) is corpus-curative, not embedding-geometric (7/10 models).
- Extension F: bg ambiguous-to-the-core (low confidence) raise Δρ_sym by +0.027 vs baseline; small n=20 effect, interpretive hint only.
- Extension E: 3 of 6 axes are pool-sensitive (rights_duties, status_contract, state_market) — Kozlowski projection yields no tradition-level invariants.

**Empirical justification of pre-registered choices** (§2.3):

- Extension H: ρ_cross saturates around K≥4 (anti-correlated at K=1). The pre-registered threshold is correct.
